In [4]:
import pandas as pd
import numpy as np
import json
import warnings
import ast
import re
import os
from tqdm.auto import tqdm

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
import ast
import re
import os
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

file_path7 = 'wikidata14.csv'
file_path9 = 'wikidata18.csv'

print("Carregando os arquivos...")
df7 = pd.read_csv(file_path7)
df9 = pd.read_csv(file_path9)

print(f"Total de registros em wikidata7.csv: {len(df7)}")
print(f"Total de registros em wikidata9.csv: {len(df9)}")

ids_7 = set(df7['id'].astype(str))
ids_9 = set(df9['id'].astype(str))

intersecao = ids_7.intersection(ids_9)
faltantes = ids_7 - ids_9

print("\n=== Conferência de IDs ===")
print(f"IDs de wikidata7 presentes em wikidata9: {len(intersecao)} ({(len(intersecao)/len(ids_7))*100:.2f}%)")
if len(faltantes) > 0:
    print(f"ATENÇÃO: Existem {len(faltantes)} IDs no wikidata7 que NÃO estão no wikidata9.")
else:
    print("PERFEITO: Todos os IDs do wikidata7 estão contidos no wikidata9!")

Carregando os arquivos...
Total de registros em wikidata7.csv: 529063
Total de registros em wikidata9.csv: 6683854

=== Conferência de IDs ===
IDs de wikidata7 presentes em wikidata9: 528186 (99.83%)
ATENÇÃO: Existem 877 IDs no wikidata7 que NÃO estão no wikidata9.


In [2]:
def criar_chave_primeiro_ultimo(nome):
    if pd.isna(nome): return ""
    nome_str = str(nome).strip()
    partes = nome_str.split()
    if len(partes) >= 2:
        return f"{partes[0]} {partes[-1]}"
    return nome_str

def criar_chave_exato(nome):
    if pd.isna(nome): return ""
    return str(nome).strip()

# aplicar as chaves na base gigante (df9) que será onde os homônimos serão buscados
df9['homonimo_key_st1'] = df9['nome'].apply(criar_chave_primeiro_ultimo)
df9['homonimo_key_st2'] = df9['nome'].apply(criar_chave_exato)

# aplicar as chaves no df7 para saber como referenciar
df7['homonimo_key_st1'] = df7['nome'].apply(criar_chave_primeiro_ultimo)
df7['homonimo_key_st2'] = df7['nome'].apply(criar_chave_exato)

counts_st1 = df9[df9['homonimo_key_st1'] != ""]['homonimo_key_st1'].value_counts().to_dict()
counts_st2 = df9[df9['homonimo_key_st2'] != ""]['homonimo_key_st2'].value_counts().to_dict()

df7['qtd_homonimos_st1'] = df7['homonimo_key_st1'].map(lambda x: counts_st1.get(x, 0))
df7['qtd_homonimos_st2'] = df7['homonimo_key_st2'].map(lambda x: counts_st2.get(x, 0))

print("=== Pessoas do wikidata7 com homônimos na base nova (wikidata9) ===")
print(f"Estratégia 1 (Primeiro + Último): {len(df7[df7['qtd_homonimos_st1'] >= 2])} pessoas do df7 possuem homônimos.")
print(f"Estratégia 2 (Nome Completo Exato): {len(df7[df7['qtd_homonimos_st2'] >= 2])} pessoas do df7 possuem homônimos.")

=== Pessoas do wikidata7 com homônimos na base nova (wikidata9) ===
Estratégia 1 (Primeiro + Último): 190697 pessoas do df7 possuem homônimos.
Estratégia 2 (Nome Completo Exato): 115398 pessoas do df7 possuem homônimos.


In [3]:
colunas_relacao = ['pai', 'mae', 'irmao', 'conjugue', 'filho', 'parente']
ids_validos_df9 = set(df9['id'].astype(str))

def calcular_estatisticas_estrategia(df_origem, coluna_qtd, nome_estrategia):
    total_linhas_finais = 0
    tamanhos_homonimos = []
    
    # filtra apenas P1 que tem pelo menos 2 ocorrências do nome no df9
    df_p1 = df_origem[df_origem[coluna_qtd] >= 2]
    
    for idx, row in df_p1.iterrows():
        qtd_homonimos_extras = row[coluna_qtd] - 1
        
        for col_rel in colunas_relacao:
            val_rel = row[col_rel]
            if pd.isna(val_rel) or str(val_rel) == 'nan': continue
            
            ids_relacionados = str(val_rel).split(';')
            for id_p2 in ids_relacionados:
                id_p2 = id_p2.strip()
                
                if id_p2 in ids_validos_df9:
                    total_linhas_finais += 1
                    tamanhos_homonimos.append(qtd_homonimos_extras)
                    
    print(f"=== ESTATÍSTICAS: {nome_estrategia} ===")
    print(f"Total de PARES (linhas esperadas no JSONL): {total_linhas_finais}")
    if tamanhos_homonimos:
        arr_stats = np.array(tamanhos_homonimos)
        print(f"Quantidade de homônimos EXTRAS por par (Tamanho da lista - 1):")
        print(f"  Mínimo: {arr_stats.min()}")
        print(f"  Máximo: {arr_stats.max()}")
        print(f"  Média:  {arr_stats.mean():.4f}")
        print(f"  Desvio: {arr_stats.std():.4f}")
        print(f"Soma total de elementos gerados nas listas: {np.sum(arr_stats + 1)}")
    else:
        print("Nenhum par seria gerado.")
    print("-" * 50)

calcular_estatisticas_estrategia(df7, 'qtd_homonimos_st1', 'Estratégia 1 (Primeiro + Último Nome)')
calcular_estatisticas_estrategia(df7, 'qtd_homonimos_st2', 'Estratégia 2 (Nome Completo Exato)')

=== ESTATÍSTICAS: Estratégia 1 (Primeiro + Último Nome) ===
Total de PARES (linhas esperadas no JSONL): 451094
Quantidade de homônimos EXTRAS por par (Tamanho da lista - 1):
  Mínimo: 1
  Máximo: 1132
  Média:  14.1344
  Desvio: 52.2687
Soma total de elementos gerados nas listas: 6827021
--------------------------------------------------
=== ESTATÍSTICAS: Estratégia 2 (Nome Completo Exato) ===
Total de PARES (linhas esperadas no JSONL): 265131
Quantidade de homônimos EXTRAS por par (Tamanho da lista - 1):
  Mínimo: 1
  Máximo: 690
  Média:  10.6845
  Desvio: 47.3400
Soma total de elementos gerados nas listas: 3097918
--------------------------------------------------


In [ ]:
# 'homonimo_key_st1' ou 'homonimo_key_st2'
CHAVE_ESCOLHIDA = 'homonimo_key_st2' 

print(f"Iniciando geração do jsonl usando '{CHAVE_ESCOLHIDA}'...")

cols_export = [
    'id', 'nome', 'genero', 'data_nascimento', 
    'profissao', 'nacionalidade', 'local_nascimento', 'endereço',
    CHAVE_ESCOLHIDA
]

def converter_para_lista(valor):
    if pd.isna(valor) or valor == '[]' or valor == '': return []
    if isinstance(valor, list): return valor
    try:
        return ast.literal_eval(valor)
    except:
        return []

for col in ['profissao', 'nacionalidade', 'local_nascimento', 'endereço']:
    df9[col] = df9[col].apply(converter_para_lista)

# extrai todos os dados do df9
dados_pessoas_dict = df9.set_index('id', drop=False)[cols_export].to_dict('index')

# homônimos são agrupados usando a base df9
grupo_homonimos = df9.groupby(CHAVE_ESCOLHIDA)['id'].apply(list).to_dict()

def get_homonym_list(nome_chave):
    ids_h = grupo_homonimos.get(nome_chave, [])
    if len(ids_h) < 2:
        return None
    
    lista_objetos = []
    for h_id in ids_h:
        if h_id in dados_pessoas_dict:
            dado = dados_pessoas_dict[h_id].copy()
            dado['id'] = h_id 
            lista_objetos.append(dado)
    return lista_objetos

registros_jsonl = []
colunas_relacao = ['pai', 'mae', 'irmao', 'conjugue', 'filho', 'parente']

# define qual coluna de quantidade olhar no df7
col_qtd = 'qtd_homonimos_st1' if CHAVE_ESCOLHIDA == 'homonimo_key_st1' else 'qtd_homonimos_st2'

# filtra df7: P1 precisa ter relações documentadas e possuir homônimos no df9
df_p1 = df7[df7[col_qtd] >= 2]

for idx, row in df_p1.iterrows():
    id_p1 = str(row['id'])
    
    # 1. Primeiro, verifique se a pessoa existe no df9
    if id_p1 not in dados_pessoas_dict:
        continue
    
    # 2. BUSCA A CHAVE NO DF9 (dados_pessoas_dict) E NÃO NO ROW (df7)
    # Isso garante que usaremos a mesma chave que gerou o grupo_homonimos
    dados_p1 = dados_pessoas_dict[id_p1].copy()
    chave_h_p1 = dados_p1[CHAVE_ESCOLHIDA] 
    
    lista_h_objetos = get_homonym_list(chave_h_p1)
    if not lista_h_objetos: continue
    
    # 3. Garantia extra (Opcional): Verificar se o ID está na lista
    ids_na_lista = [str(h['id']) for h in lista_h_objetos]
    if id_p1 not in ids_na_lista:
        # Se por algum erro bizarro ainda não estiver, inserimos manualmente
        lista_h_objetos.append(dados_p1)
    
    for col_rel in colunas_relacao:
        val_rel = row[col_rel]
        if pd.isna(val_rel) or str(val_rel) == 'nan': continue
        
        ids_relacionados = str(val_rel).split(';')
        
        for id_p2 in ids_relacionados:
            id_p2 = id_p2.strip()
            
            # busca dados do P2 na base completa (df9)
            if id_p2 in dados_pessoas_dict:
                dados_p2 = dados_pessoas_dict[id_p2].copy()
                
                registro = {
                    "Dados_pessoa_1": dados_p1,
                    "Dados_pessoa_2": dados_p2,
                    "Lista_homonimos": lista_h_objetos,
                    "Parentesco": col_rel
                }
                registros_jsonl.append(registro)

output_json = 'wikidata_pessoa_homonimo.jsonl'

with open(output_json, 'w', encoding='utf-8') as f:
    for item in registros_jsonl:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processo concluído.")

Iniciando geração do jsonl usando 'homonimo_key_st2'...
Salvando arquivo wikidata_pessoa_homonimo.jsonl com 264507 pares finais...
Processo concluído com sucesso!


In [ ]:
ARQUIVO_ORIGINAL = 'wikidata_pessoa_homonimo.jsonl'
ARQUIVO_TEMPORARIO = 'wikidata_pessoa_homonimo_temp.jsonl'

def is_invalid_value(val):
    """Verifica se o valor em texto bate com algum dos padrões de sujeira."""
    if not isinstance(val, str):
        return True
    
    val = val.strip()
    
    if val.isdigit(): return True                          # 1. Apenas números
    if re.match(r'^Q\d+$', val): return True               # 2. Código Wikidata
    if val.startswith('_:node'): return True               # 3. Nós RDF
        
    return False

def clean_person_dict(person_data):
    """Limpa as listas de atributos de um dicionário de pessoa."""
    campos_para_limpar = ['profissao', 'nacionalidade', 'local_nascimento', 'endereço']
    for campo in campos_para_limpar:
        if campo in person_data and isinstance(person_data[campo], list):
            person_data[campo] = [v for v in person_data[campo] if not is_invalid_value(v)]
    return person_data

linhas_mantidas = 0
linhas_removidas = 0

print(f"[{ARQUIVO_ORIGINAL}] Iniciando limpeza...")

with open(ARQUIVO_ORIGINAL, 'r', encoding='utf-8') as fin, \
     open(ARQUIVO_TEMPORARIO, 'w', encoding='utf-8') as fout:
    
    total_lines = sum(1 for _ in fin)
    fin.seek(0)
    
    for line in tqdm(fin, total=total_lines, desc="Limpando JSONL"):
        if not line.strip(): continue
        row = json.loads(line)
        
        # Limpa os dados das pessoas principais e homônimos
        if 'Dados_pessoa_1' in row: row['Dados_pessoa_1'] = clean_person_dict(row['Dados_pessoa_1'])
        if 'Dados_pessoa_2' in row: row['Dados_pessoa_2'] = clean_person_dict(row['Dados_pessoa_2'])
            
        if 'Lista_homonimos' in row and isinstance(row['Lista_homonimos'], list):
            row['Lista_homonimos'] = [clean_person_dict(h) for h in row['Lista_homonimos']]
            
            # Valida tamanho da lista
            if len(row['Lista_homonimos']) >= 2:
                fout.write(json.dumps(row, ensure_ascii=False) + '\n')
                linhas_mantidas += 1
            else:
                linhas_removidas += 1
        else:
            linhas_removidas += 1

# Substitui o arquivo original pelo arquivo limpo
os.replace(ARQUIVO_TEMPORARIO, ARQUIVO_ORIGINAL)
print(f"✔️ Linhas salvas: {linhas_mantidas} | ❌ Linhas descartadas: {linhas_removidas}")

print("\nCarregando dados limpos para gerar as estatísticas...")
registros_jsonl = []
with open(ARQUIVO_ORIGINAL, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            registros_jsonl.append(json.loads(line))

pessoas_ids = set()
nacionalidades_distintas = set()
enderecos_distintos = set()
profissoes_distintas = set()
locais_nasc_distintos = set()

tamanhos_homonimos_extras = []
tamanhos_homonimos_total = []

def extrair_distintos(valor, conjunto):
    if isinstance(valor, (list, np.ndarray)):
        for item in valor:
            if pd.notna(item) and str(item).strip().lower() not in ['nan', 'none', '']:
                conjunto.add(str(item).strip())
        return
    
    if pd.isna(valor) or str(valor).strip().lower() in ['nan', 'none', '']:
        return
    
    if isinstance(valor, str) and ';' in valor:
        itens = [item.strip() for item in valor.split(';')]
        for item in itens:
            if item and item.lower() not in ['nan', 'none', '']: 
                conjunto.add(item)
    else:
        conjunto.add(str(valor).strip())

def catalogar_pessoa(dados_pessoa):
    if not dados_pessoa: return
    pessoas_ids.add(str(dados_pessoa.get('id')))
    extrair_distintos(dados_pessoa.get('nacionalidade'), nacionalidades_distintas)
    extrair_distintos(dados_pessoa.get('endereço'), enderecos_distintos)
    extrair_distintos(dados_pessoa.get('profissao'), profissoes_distintas)
    extrair_distintos(dados_pessoa.get('local_nascimento'), locais_nasc_distintos)

for registro in registros_jsonl:
    catalogar_pessoa(registro.get('Dados_pessoa_1'))
    catalogar_pessoa(registro.get('Dados_pessoa_2'))
    
    lista_h = registro.get('Lista_homonimos', [])
    for homonimo in lista_h:
        catalogar_pessoa(homonimo)
        
    tam_lista = len(lista_h)
    tamanhos_homonimos_total.append(tam_lista)
    tamanhos_homonimos_extras.append(tam_lista - 1)

arr_extras = np.array(tamanhos_homonimos_extras)

qtd_pares = len(registros_jsonl)
qtd_pessoas = len(pessoas_ids)
media_h_extras = arr_extras.mean() if qtd_pares > 0 else 0
desvio_h_extras = arr_extras.std() if qtd_pares > 0 else 0
soma_h_total = sum(tamanhos_homonimos_total)

print("\n" + "=" * 60)
print("PAINEL DE ESTATÍSTICAS DO EXPERIMENTO (PÓS-LIMPEZA)".center(60))
print("=" * 60)
print(f"Quantidade de pares:                          {qtd_pares}")
print(f"Qtd pessoas distintas experimento:            {qtd_pessoas}")
print(f"Media (lista homonimo - 1):                   {media_h_extras:.4f}")
print(f"Desvio (lista homonimo - 1):                  {desvio_h_extras:.4f}")
print(f"Somatorio (lista homonimo):                   {soma_h_total}")
print("-" * 60)
print(f"Qtd nacionalidades distintas experimento:     {len(nacionalidades_distintas)}")
print(f"Qtd enderecos distintos experimento:          {len(enderecos_distintos)}")
print(f"Qtd profissoes distintas experimento:         {len(profissoes_distintas)}")
print(f"Qtd local nascimento distintas experimento:   {len(locais_nasc_distintos)}")
print("=" * 60)

[wikidata_pessoa_homonimo.jsonl] Iniciando limpeza...


Limpando JSONL:   0%|          | 0/264507 [00:00<?, ?it/s]

✔️ Linhas salvas: 264507 | ❌ Linhas descartadas: 0

Carregando dados limpos para gerar as estatísticas...

    PAINEL DE ESTATÍSTICAS DO EXPERIMENTO (PÓS-LIMPEZA)     
Quantidade de pares:                          264507
Qtd pessoas distintas experimento:            531296
Media (lista homonimo - 1):                   10.6935
Desvio (lista homonimo - 1):                  47.3905
Somatorio (lista homonimo):                   3093024
------------------------------------------------------------
Qtd nacionalidades distintas experimento:     809
Qtd enderecos distintos experimento:          10747
Qtd profissoes distintas experimento:         6328
Qtd local nascimento distintas experimento:   52226
